# Goalkeeper Positioning — 03: Counterfactual Analysis

This is what the model is actually for. Given a trained `V(x, g) = P(goal | x, g)`, we hold the non-GK context (shooter, ball, attacking teammates, defenders) fixed and **sweep the goalkeeper across a 2-D grid of candidate positions**. The danger map `V(x, ·)` over that grid tells us:

- `g* = argmin_g V(x, g)` — the model's preferred goalkeeper position for this shot
- `regret = V(x, g_actual) - V(x, g*)` — how much danger the goalkeeper "left on the table" by being where they were

High-regret examples are the ones to scrutinize. Are they real positioning errors, or does the model just dislike the actual position because it's seen too few similar configurations?

**Sections:**
1. Setup
2. The sweep
3. Visualization
4. Curated examples on the test split

This notebook replaces `src/analysis/counterfactual.py`.

**Prerequisites:** Notebooks 01 and 02 have been run (a checkpoint exists at `models/checkpoints/baseline_v1/best.pt`).

## 1. Setup

In [ ]:
from __future__ import annotations

import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from mplsoccer import Pitch

REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

# Reuse the data + model primitives from notebooks 01 / 02.
from src.data.rasterize import GRID_H, GRID_W, SIGMA, rasterize_shot, render_gaussian
from src.models.danger_cnn import DangerCNN

SHOTS_PATH = REPO_ROOT / "data/raw/shots_master_df.csv"
FREEZE_PATH = REPO_ROOT / "data/raw/freeze_master_df.csv"
TEST_MANIFEST = REPO_ROOT / "data/processed/splits/test_shot_ids.csv"
DEFAULT_CKPT = REPO_ROOT / "models/checkpoints/baseline_v1/best.pt"
OUTPUT_DIR = REPO_ROOT / "results/counterfactual"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"REPO_ROOT      = {REPO_ROOT}")
print(f"DEFAULT_CKPT   = {DEFAULT_CKPT}  exists={DEFAULT_CKPT.exists()}")

## 2. The sweep

The 2-D grid of candidate GK positions covers the 6-yard area in front of goal:
- `x ∈ [110, 120]`, step 0.5 m (1 m past the 18-yard line through the goal line)
- `y ∈ [30, 50]`, step 0.5 m (centered on the goal mouth `y ∈ [36, 44]`, with breathing room)

For each grid point we replace channel 4 (goalkeeper) of the rasterized input with a fresh Gaussian at that position, leave the other 4 channels untouched, and run a single batched forward pass over all `n_x × n_y` candidates at once.

### 2.1 Defaults and helpers

In [ ]:
DEFAULT_GRID_X = np.linspace(110.0, 120.0, 21)   # 21 points, 0.5 m spacing
DEFAULT_GRID_Y = np.linspace(30.0, 50.0, 41)     # 41 points, 0.5 m spacing


def _select_device():
    if torch.cuda.is_available():
        return torch.device("cuda")
    if torch.backends.mps.is_available():
        return torch.device("mps")
    return torch.device("cpu")


def _actual_gk_pos(freeze_rows: pd.DataFrame) -> tuple[float, float]:
    gk = freeze_rows[
        (freeze_rows["position_name"] == "Goalkeeper") & (~freeze_rows["teammate"])
    ].iloc[0]
    return float(gk["x"]), float(gk["y"])


def _load_model(checkpoint_path: Path, device: torch.device) -> DangerCNN:
    ckpt = torch.load(checkpoint_path, map_location="cpu", weights_only=False)
    model = DangerCNN().to(device)
    model.load_state_dict(ckpt["model_state_dict"])
    model.eval()
    return model

### 2.2 `sweep_gk_positions`

In [ ]:
def sweep_gk_positions(
    model: DangerCNN,
    shot_row: pd.Series,
    freeze_rows: pd.DataFrame,
    gk_grid_x=None,
    gk_grid_y=None,
    device=None,
) -> dict:
    """Hold non-GK context fixed; sweep GK over a 2-D grid; return the danger map."""
    if gk_grid_x is None: gk_grid_x = DEFAULT_GRID_X
    if gk_grid_y is None: gk_grid_y = DEFAULT_GRID_Y
    if device is None: device = next(model.parameters()).device

    base = rasterize_shot(shot_row, freeze_rows)            # (5, 80, 60)
    context = base[:4].unsqueeze(0).to(device)              # (1, 4, 80, 60)

    n_y, n_x = len(gk_grid_y), len(gk_grid_x)
    n = n_y * n_x

    gk_channels = np.empty((n, GRID_H, GRID_W), dtype=np.float32)
    k = 0
    for gy in gk_grid_y:
        for gx in gk_grid_x:
            gk_channels[k] = render_gaussian(GRID_H, GRID_W, float(gx), float(gy), SIGMA)
            k += 1
    gk_tensor = torch.from_numpy(gk_channels).unsqueeze(1).to(device)        # (N, 1, 80, 60)
    batch = torch.cat([context.expand(n, -1, -1, -1), gk_tensor], dim=1)     # (N, 5, 80, 60)

    model.eval()
    with torch.no_grad():
        probs = torch.sigmoid(model.forward_logits(batch)).cpu().numpy()
        actual_v = float(
            torch.sigmoid(model.forward_logits(base.unsqueeze(0).to(device))).cpu().item()
        )
    danger_grid = probs.reshape(n_y, n_x)

    yi, xi = np.unravel_index(np.argmin(danger_grid), danger_grid.shape)
    optimal_gk_pos = (float(gk_grid_x[xi]), float(gk_grid_y[yi]))
    optimal_v = float(danger_grid[yi, xi])

    return {
        "danger_grid":      danger_grid,
        "gk_grid_x":        gk_grid_x,
        "gk_grid_y":        gk_grid_y,
        "actual_gk_pos":    _actual_gk_pos(freeze_rows),
        "actual_v":         actual_v,
        "optimal_gk_pos":   optimal_gk_pos,
        "optimal_v":        optimal_v,
        "shot_id":          shot_row["id"],
        "is_goal":          int(shot_row.get("outcome_name") == "Goal"),
    }

## 3. Visualization

A 4-panel layout per shot:
1. Pitch view with players, the actual GK position, and the optimal `g*`
2. `V(x, g)` heatmap over the GK grid
3. Same heatmap as a contour plot
4. Text summary (positions, V values, regret)

### 3.1 `visualize_sweep`

In [ ]:
def visualize_sweep(sweep_result, shot_row, freeze_rows, output_path):
    grid_x = sweep_result["gk_grid_x"]
    grid_y = sweep_result["gk_grid_y"]
    danger = sweep_result["danger_grid"]
    a_gx, a_gy = sweep_result["actual_gk_pos"]
    o_gx, o_gy = sweep_result["optimal_gk_pos"]
    is_goal = bool(sweep_result["is_goal"])
    regret = sweep_result["actual_v"] - sweep_result["optimal_v"]

    fig = plt.figure(figsize=(15, 11))
    gs = fig.add_gridspec(2, 2, hspace=0.28, wspace=0.22)

    # Panel 1: pitch
    pitch = Pitch(pitch_type="statsbomb", half=True, line_color="black")
    ax1 = fig.add_subplot(gs[0, 0])
    pitch.draw(ax=ax1)

    sx, sy = float(shot_row["x"]), float(shot_row["y"])
    teammates = freeze_rows[(freeze_rows["teammate"]) & (freeze_rows["position_name"] != "Goalkeeper")]
    defenders = freeze_rows[(~freeze_rows["teammate"]) & (freeze_rows["position_name"] != "Goalkeeper")]

    pitch.scatter(teammates["x"], teammates["y"], ax=ax1, s=80, color="#1f77b4",
                  alpha=0.7, edgecolors="black", linewidths=0.5, label="Attackers", zorder=3)
    pitch.scatter(defenders["x"], defenders["y"], ax=ax1, s=80, color="#7f7f7f",
                  alpha=0.7, edgecolors="black", linewidths=0.5, label="Defenders", zorder=3)
    pitch.scatter([sx], [sy], ax=ax1, s=240, color="red", marker="*",
                  edgecolors="black", linewidths=0.5, label="Shooter", zorder=5)
    pitch.scatter([a_gx], [a_gy], ax=ax1, s=180, color="#2ca02c", marker="o",
                  edgecolors="black", linewidths=1.0, label="Actual GK", zorder=6)
    pitch.scatter([o_gx], [o_gy], ax=ax1, s=320, color="gold", marker="*",
                  edgecolors="black", linewidths=1.0, label="Optimal g*", zorder=7)
    ax1.set_title(f"Shot {sweep_result['shot_id']} — {'GOAL' if is_goal else 'no goal'}")
    ax1.legend(loc="lower left", fontsize=8, framealpha=0.9)

    # Panel 2: heatmap
    ax2 = fig.add_subplot(gs[0, 1])
    im = ax2.pcolormesh(grid_x, grid_y, danger, cmap="Reds", shading="auto")
    fig.colorbar(im, ax=ax2, label="V(x, g)")
    ax2.scatter([a_gx], [a_gy], s=180, color="#2ca02c", marker="o",
                edgecolors="black", linewidths=1.0, label="Actual", zorder=5)
    ax2.scatter([o_gx], [o_gy], s=280, color="gold", marker="*",
                edgecolors="black", linewidths=1.0, label="g*", zorder=6)
    for gy_post in (36.0, 44.0):
        ax2.axhline(gy_post, color="black", linestyle=":", linewidth=0.8, alpha=0.5)
    ax2.set_xlabel("GK x (m)"); ax2.set_ylabel("GK y (m)")
    ax2.set_title("V(x, g) heatmap")
    ax2.legend(loc="upper right", fontsize=8, framealpha=0.9)
    ax2.invert_yaxis()  # match StatsBomb (y=0 at top)

    # Panel 3: contour
    ax3 = fig.add_subplot(gs[1, 0])
    cs = ax3.contourf(grid_x, grid_y, danger, levels=15, cmap="Reds")
    ax3.contour(grid_x, grid_y, danger, levels=8, colors="black", linewidths=0.5, alpha=0.5)
    fig.colorbar(cs, ax=ax3, label="V(x, g)")
    ax3.scatter([a_gx], [a_gy], s=180, color="#2ca02c", marker="o",
                edgecolors="black", linewidths=1.0, label="Actual", zorder=5)
    ax3.scatter([o_gx], [o_gy], s=280, color="gold", marker="*",
                edgecolors="black", linewidths=1.0, label="g*", zorder=6)
    for gy_post in (36.0, 44.0):
        ax3.axhline(gy_post, color="black", linestyle=":", linewidth=0.8, alpha=0.5)
    ax3.set_xlabel("GK x (m)"); ax3.set_ylabel("GK y (m)")
    ax3.set_title("V(x, g) contour")
    ax3.legend(loc="upper right", fontsize=8, framealpha=0.9)
    ax3.invert_yaxis()

    # Panel 4: text summary
    ax4 = fig.add_subplot(gs[1, 1])
    ax4.axis("off")
    xg = shot_row.get("shot_statsbomb_xg", np.nan)
    xg_str = f"{xg:.4f}" if pd.notna(xg) else "n/a"
    lines = [
        f"shot_id        {sweep_result['shot_id']}",
        f"outcome        {'GOAL' if is_goal else 'no goal'}",
        f"StatsBomb xG   {xg_str}",
        "",
        f"actual GK pos  ({a_gx:.1f}, {a_gy:.1f})",
        f"actual V       {sweep_result['actual_v']:.4f}",
        "",
        f"optimal g*     ({o_gx:.1f}, {o_gy:.1f})",
        f"optimal V      {sweep_result['optimal_v']:.4f}",
        "",
        f"regret         {regret:+.4f}",
        "               (actual_v - optimal_v)",
    ]
    ax4.text(0.05, 0.95, "\n".join(lines), fontsize=12, family="monospace",
             va="top", transform=ax4.transAxes)

    output_path = Path(output_path)
    output_path.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(output_path, dpi=120, bbox_inches="tight")
    plt.show()
    plt.close(fig)

## 4. Curated examples on the test split

We're not interested in random shots — we want a few from each of:
- **high-regret-goal**: the GK was badly positioned *and* the shot went in
- **high-regret-no-goal**: the GK was badly positioned but got away with it
- **low-regret**: the GK was already close to optimal (sanity-check that the model agrees with good positioning)
- **random**: a baseline mix

To pick these, we sweep a sample of `sample_size` test shots, compute regret for each, then bucket them.

### 4.1 Bucket-picker

In [ ]:
def _pick_examples(regret_df: pd.DataFrame, per_bucket: int, seed: int) -> pd.DataFrame:
    """Pick `per_bucket` examples each from 4 buckets. No shot is picked twice."""
    chosen_parts = []
    seen: set = set()

    def _take(df, label):
        df = df[~df["shot_id"].isin(seen)]
        if len(df) == 0:
            return
        df = df.head(per_bucket).assign(category=label)
        chosen_parts.append(df)
        seen.update(df["shot_id"].tolist())

    _take(regret_df[regret_df["is_goal"] == 1].sort_values("regret", ascending=False),
          "high-regret-goal")
    _take(regret_df[regret_df["is_goal"] == 0].sort_values("regret", ascending=False),
          "high-regret-no-goal")
    _take(regret_df.sort_values("regret", ascending=True), "low-regret")
    remaining = regret_df[~regret_df["shot_id"].isin(seen)]
    if len(remaining) > 0:
        chosen_parts.append(
            remaining.sample(n=min(per_bucket, len(remaining)), random_state=seed)
            .assign(category="random")
        )
    return pd.concat(chosen_parts, ignore_index=True)

### 4.2 Driver — sweep, bucket, visualize

In [ ]:
def run_sweep_on_examples(
    checkpoint_path,
    shot_ids=None,
    n_examples: int = 8,
    output_dir: Path = OUTPUT_DIR,
    sample_size: int = 200,
    seed: int = 42,
) -> pd.DataFrame:
    """Run sweep on a curated set of test shots; write 4-panel PNGs + summary.csv."""
    checkpoint_path = Path(checkpoint_path)
    if not checkpoint_path.is_absolute():
        checkpoint_path = REPO_ROOT / checkpoint_path
    output_dir = Path(output_dir)
    if not output_dir.is_absolute():
        output_dir = REPO_ROOT / output_dir
    output_dir.mkdir(parents=True, exist_ok=True)

    device = _select_device()
    model = _load_model(checkpoint_path, device)
    print(f"Loaded {checkpoint_path.name} on {device}")

    print("Loading shot/freeze masters...")
    shots_df = pd.read_csv(SHOTS_PATH)
    freeze_df = pd.read_csv(FREEZE_PATH)

    if shot_ids is None:
        rng = np.random.default_rng(seed)
        test_ids = pd.read_csv(TEST_MANIFEST)["id"].tolist()
        idx = rng.choice(len(test_ids), size=min(sample_size, len(test_ids)), replace=False)
        sample_ids = [test_ids[i] for i in idx]
        print(f"Sweeping {len(sample_ids)} test shots to compute regret...")
        results = []
        for i, sid in enumerate(sample_ids, start=1):
            shot = shots_df[shots_df["id"] == sid].iloc[0]
            fr = freeze_df[freeze_df["id"] == sid]
            results.append(sweep_gk_positions(model, shot, fr, device=device))
            if i % 25 == 0 or i == len(sample_ids):
                print(f"  {i}/{len(sample_ids)} swept")
        regret_df = pd.DataFrame([
            {
                "shot_id":   r["shot_id"],
                "is_goal":   r["is_goal"],
                "actual_v":  r["actual_v"],
                "optimal_v": r["optimal_v"],
                "regret":    r["actual_v"] - r["optimal_v"],
            }
            for r in results
        ])
        per_bucket = max(1, n_examples // 4)
        chosen = _pick_examples(regret_df, per_bucket=per_bucket, seed=seed)
        results_by_id = {r["shot_id"]: r for r in results}
    else:
        print(f"Sweeping {len(shot_ids)} user-specified shots...")
        results_by_id = {}
        records = []
        for sid in shot_ids:
            shot = shots_df[shots_df["id"] == sid].iloc[0]
            fr = freeze_df[freeze_df["id"] == sid]
            r = sweep_gk_positions(model, shot, fr, device=device)
            results_by_id[sid] = r
            records.append({
                "shot_id":   sid,
                "is_goal":   r["is_goal"],
                "actual_v":  r["actual_v"],
                "optimal_v": r["optimal_v"],
                "regret":    r["actual_v"] - r["optimal_v"],
                "category":  "user",
            })
        chosen = pd.DataFrame(records)

    print()
    header = (
        f"{'category':<22}{'shot_id':>14}{'goal':>6}"
        f"{'actual_v':>11}{'optimal_v':>11}{'regret':>10}"
    )
    print(header)
    print("-" * len(header))
    for _, row in chosen.iterrows():
        sid = row["shot_id"]
        r = results_by_id[sid]
        shot = shots_df[shots_df["id"] == sid].iloc[0]
        fr = freeze_df[freeze_df["id"] == sid]
        png_path = output_dir / f"{row['category']}_{sid}.png"
        visualize_sweep(r, shot, fr, png_path)
        print(
            f"{row['category']:<22}{str(sid):>14}{int(row['is_goal']):>6}"
            f"{row['actual_v']:>11.4f}{row['optimal_v']:>11.4f}"
            f"{row['regret']:>+10.4f}"
        )

    summary_path = output_dir / "summary.csv"
    chosen.to_csv(summary_path, index=False)
    print(f"\nSaved {len(chosen)} visualizations and summary.csv to {output_dir}")
    return chosen

### 4.3 Run

8 examples (2 per bucket) by default. Each call produces a 4-panel PNG per chosen shot, plus a `summary.csv`. Set `n_examples=12` or `16` for a more thorough look.

In [ ]:
summary = run_sweep_on_examples(DEFAULT_CKPT, n_examples=8)
summary

### 4.4 Inspect a single shot ad hoc

If you want to drill into one specific shot (e.g. one a teammate flagged), pass its ID directly:

In [ ]:
# Example: pick the highest-regret-goal from the curated set above
if not summary.empty:
    interesting = summary.sort_values("regret", ascending=False).iloc[0]
    print(f"Drilling into shot {interesting['shot_id']} "
          f"(category={interesting['category']}, regret={interesting['regret']:+.4f})")
    summary_one = run_sweep_on_examples(
        DEFAULT_CKPT, shot_ids=[interesting["shot_id"]],
        output_dir=OUTPUT_DIR / "ad_hoc",
    )
    summary_one

---

**Done.** Heatmaps and `summary.csv` are under `results/counterfactual/`.

**Caveats to keep in mind when interpreting the heatmaps:**
- `V(x, g)` is what the model *predicts*, not what would actually happen. High regret on a single shot is interesting; consistent patterns across many shots are more trustworthy.
- The GK grid is restricted to the 6-yard area. Anything outside that range is silently absent from the optimum — fine for everyday positioning, less fine if you want to ask "should the keeper rush out?"
- The sweep only varies `g`; the rest of the freeze frame is frozen. In reality, defenders adjust to the keeper.
- The training set is men's 2015/16 top-5 leagues. Sweeps on `transfer_*` shots run, but interpret with that domain shift in mind.